# LAB 04 — Construye el RAG completo

**Documents → Chunking → Embeddings → Vector Store → Query → Retrieval → Top-K → Context Assembly → Prompt → LLM → Answer**

**Tiempo sugerido:** 60 minutos · **Requiere proveedor configurado** (OpenAI u Ollama)

Al terminar tendrás un RAG funcional sobre tu documento, con citas a las fuentes.
No estamos usando magia: estamos conectando componentes simples.

In [ ]:
# Instalación de dependencias: instala en este kernel los paquetes que falten (si ya están, no hace nada)
import sys, shutil, subprocess, importlib, importlib.util

PAQUETES = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "tiktoken": "tiktoken",
            "openai": "openai", "dotenv": "python-dotenv", "matplotlib": "matplotlib"}

def _faltantes():
    importlib.invalidate_caches()
    return [pip for mod, pip in PAQUETES.items() if importlib.util.find_spec(mod) is None]

faltan = _faltantes()
if faltan:
    print("Instalando:", ", ".join(faltan), "→", sys.executable, "(puede tardar 1-2 minutos)")
    intentos = []
    if shutil.which("uv"):
        intentos.append(["uv", "pip", "install", "--python", sys.executable, *faltan])
    intentos.append([sys.executable, "-m", "pip", "install", "--quiet", "--retries", "1", "--timeout", "20", *faltan])
    for cmd in intentos:
        if cmd[0] == sys.executable and subprocess.run(cmd[:3] + ["--version"], capture_output=True).returncode != 0:
            subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], capture_output=True)
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0 and not _faltantes():
            break
        print(f"⚠️ Falló el intento con {cmd[0]}; probando otra vía...\n", (r.stderr or r.stdout)[-800:])
    faltan = _faltantes()
    if faltan:
        raise ModuleNotFoundError(
            f"No se pudieron instalar: {', '.join(faltan)} en {sys.executable}\n"
            "Revisa tu conexión a internet. Solución: en la terminal, dentro del repositorio, ejecuta `uv sync`; luego en VS Code elige "
            "Select Kernel → Python Environments → .venv (la del repositorio) y reinicia el kernel."
        )
print("✅ Dependencias listas:", sys.executable)

In [ ]:
# Configuración común: localiza el repositorio, carga .env y el documento de trabajo
import os
import json
from pathlib import Path
from dotenv import load_dotenv

def encontrar_raiz():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "data").is_dir() and (c / "pyproject.toml").exists():
            return c
    raise FileNotFoundError("No encontré la carpeta data/. Abre el notebook desde el repositorio del curso.")

ROOT = encontrar_raiz()
load_dotenv(ROOT / ".env")
DATA_DIR = ROOT / "data"

DOCUMENTOS = {
    "cafeteria":   ("cafeteria_nebula_manual.md",      "cafeteria_preguntas.json"),
    "streamly":    ("streamly_api_docs.md",            "streamly_preguntas.json"),
    "veterinaria": ("clinica_veterinaria_huellitas.md", "veterinaria_preguntas.json"),
    "mi_documento": ("mi_documento.md",                 None),  # genera el tuyo con los prompts de alumnos/02_material_del_alumno.md
}

def cargar_documento(nombre):
    archivo, _ = DOCUMENTOS[nombre]
    return (DATA_DIR / archivo).read_text(encoding="utf-8")

def cargar_preguntas(nombre):
    _, archivo = DOCUMENTOS[nombre]
    if archivo is None:
        return []
    return json.loads((DATA_DIR / archivo).read_text(encoding="utf-8"))

print("Repositorio:", ROOT)

In [ ]:
# 👉 Elige el documento con el que vas a trabajar
DOCUMENTO = "cafeteria"   # "cafeteria" | "streamly" | "veterinaria" | "mi_documento"

texto = cargar_documento(DOCUMENTO)
print(f"Documento: {DOCUMENTO} — {len(texto):,} caracteres, {len(texto.split()):,} palabras")
print(texto[:500], "...")

In [ ]:
# Funciones de chunking (las construiste en el LAB 02)
import re

def chunk_text(text, chunk_size=500, overlap=100):
    if overlap >= chunk_size:
        raise ValueError("overlap debe ser menor que chunk_size")
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

def chunk_by_section(text, max_chars=1200):
    """Divide por encabezados Markdown y subdivide secciones largas por párrafos."""
    resultado, seccion, buffer = [], "Inicio", []

    def vaciar():
        cuerpo = "\n".join(buffer).strip()
        if not cuerpo:
            return
        parrafos, actual = [p for p in cuerpo.split("\n\n") if p.strip()], ""
        for p in parrafos:
            if actual and len(actual) + len(p) > max_chars:
                resultado.append({"seccion": seccion, "texto": f"{seccion}\n{actual.strip()}"})
                actual = ""
            actual += p + "\n\n"
        if actual.strip():
            resultado.append({"seccion": seccion, "texto": f"{seccion}\n{actual.strip()}"})

    for linea in text.splitlines():
        m = re.match(r"^#{1,6}\s+(.*)", linea)
        if m:
            vaciar()
            seccion, buffer = m.group(1).strip(), []
        else:
            buffer.append(linea)
    vaciar()
    return resultado

In [ ]:
# Proveedor híbrido: OpenAI (API) u Ollama (local). Se configura en .env con LLM_PROVEEDOR.
# Ollama expone un endpoint compatible con la API de OpenAI, así que usamos el mismo cliente.
import numpy as np
from openai import OpenAI

PROVEEDOR = os.getenv("LLM_PROVEEDOR", "openai").lower()

if PROVEEDOR == "openai":
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY en .env (o cambia LLM_PROVEEDOR=ollama).")
    client = OpenAI()
    EMBEDDING_MODEL = "text-embedding-3-small"
    CHAT_MODEL = "gpt-4.1-mini"
elif PROVEEDOR == "ollama":
    client = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), api_key="ollama")
    EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
    CHAT_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "llama3.2")
else:
    raise ValueError("LLM_PROVEEDOR debe ser 'openai' u 'ollama'")

# Nota: los nombres de modelos cambian con el tiempo; verifica la documentación vigente del proveedor.
print(f"Proveedor: {PROVEEDOR} | embeddings: {EMBEDDING_MODEL} | chat: {CHAT_MODEL}")

_cache_embeddings = {}

def get_embeddings(textos, batch_size=64):
    """Devuelve una matriz (n_textos, dimensiones). Cachea resultados para no pagar dos veces."""
    faltantes = [t for t in dict.fromkeys(textos) if t not in _cache_embeddings]
    for i in range(0, len(faltantes), batch_size):
        lote = faltantes[i:i + batch_size]
        resp = client.embeddings.create(model=EMBEDDING_MODEL, input=lote)
        for t, item in zip(lote, resp.data):
            _cache_embeddings[t] = item.embedding
    return np.array([_cache_embeddings[t] for t in textos], dtype=float)

## 1. Vector store en memoria

Un vector store guarda **embedding + texto + metadata** y permite buscar por similitud.

### ✏️ Ejercicio 1 — completa `SimpleVectorStore`

- `add(textos, metadatas)`: calcula embeddings y guarda todo.
- `search(consulta, k, filtro=None)`: devuelve los `k` resultados más similares como diccionarios
  `{"id", "score", "texto", "metadata"}`. Si `filtro` es un dict (p. ej. `{"source": "cafeteria"}`),
  solo considera los elementos cuya metadata coincida.

In [ ]:
class SimpleVectorStore:
    def __init__(self):
        self.textos, self.metadatas = [], []
        self.vectores = None   # matriz (n, dimensiones)

    def add(self, textos, metadatas):
        nuevos = get_embeddings(textos)
        self.vectores = nuevos if self.vectores is None else np.vstack([self.vectores, nuevos])
        self.textos.extend(textos)
        self.metadatas.extend(metadatas)

    def search(self, consulta, k=3, filtro=None):
        q = get_embeddings([consulta])[0]
        # TODO 1: calcula el coseno entre q y cada vector (puedes hacerlo vectorizado)
        # TODO 2: si hay filtro, descarta (score = -inf) los que no coincidan
        # TODO 3: devuelve los k mejores como lista de dicts {"id", "score", "texto", "metadata"}
        ...

    def __len__(self):
        return len(self.textos)

## 2. Ingesta: documento → chunks + metadata → vector store

In [ ]:
store = SimpleVectorStore()

def ingestar(nombre_doc, store, max_chars=1200):
    secciones = chunk_by_section(cargar_documento(nombre_doc), max_chars=max_chars)
    textos = [c["texto"] for c in secciones]
    metas = [{"source": nombre_doc, "section": c["seccion"], "chunk": i} for i, c in enumerate(secciones)]
    store.add(textos, metas)
    return len(textos)

print("Chunks indexados:", ingestar(DOCUMENTO, store))

In [ ]:
# ✅ Autoverificación del vector store
r = store.search("horario", k=2)
assert len(r) == 2 and r[0]["score"] >= r[1]["score"]
assert {"id", "score", "texto", "metadata"} <= set(r[0])
print("✅ Ejercicio 1 correcto")
for x in r:
    print(round(x["score"], 3), x["metadata"])

## 3. Context assembly

### ✏️ Ejercicio 2 — `construir_contexto`

Une los resultados en un solo texto. Cada fragmento debe ir precedido de una etiqueta con su id y sección,
por ejemplo `[3] (sección: 6. Pedidos para llevar...)`, para que el modelo pueda **citar** sus fuentes.

In [ ]:
def construir_contexto(resultados):
    # TODO
    ...

In [ ]:
# ✅ Autoverificación
ctx = construir_contexto(store.search("precios", k=2))
assert ctx.count("[") >= 2 and "sección" in ctx
print("✅ Ejercicio 2 correcto\n")
print(ctx[:600])

## 4. Prompt con grounding

### ✏️ Ejercicio 3 — `construir_mensajes`

Escribe el **system prompt**. Debe pedir al modelo:
1. Responder **solo** con información del contexto.
2. Citar los ids de los fragmentos usados, p. ej. `[3]`.
3. Si la respuesta no está en el contexto, responder exactamente: *"No encuentro esa información en los documentos."*
4. Si hay información contradictoria, preferir la más reciente y mencionarlo.

In [ ]:
NO_SE = "No encuentro esa información en los documentos."

SYSTEM_PROMPT = f"""
TODO: escribe aquí tus instrucciones
"""

def construir_mensajes(pregunta, contexto):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA:\n{pregunta}"},
    ]

## 5. Generación y RAG completo

In [ ]:
def generar(mensajes):
    resp = client.chat.completions.create(model=CHAT_MODEL, messages=mensajes, temperature=0)
    return resp.choices[0].message.content, resp.usage

def rag(pregunta, k=3, filtro=None, verbose=True):
    resultados = store.search(pregunta, k=k, filtro=filtro)       # Retrieval + Top-K
    contexto = construir_contexto(resultados)                     # Context assembly
    mensajes = construir_mensajes(pregunta, contexto)             # Prompt
    respuesta, uso = generar(mensajes)                            # LLM
    if verbose:
        print("❓", pregunta)
        print("📚 Recuperado:", [(r["id"], r["metadata"]["section"][:30], round(r["score"], 3)) for r in resultados])
        print("💬", respuesta)
        if uso:
            print(f"🔢 tokens: entrada={uso.prompt_tokens}, salida={uso.completion_tokens}\n")
    return {"pregunta": pregunta, "respuesta": respuesta, "resultados": resultados}

In [ ]:
preguntas = cargar_preguntas(DOCUMENTO)
for p in preguntas[:3]:
    rag(p["pregunta"])

## 6. Preguntas fuera del conocimiento

Un buen RAG debe **admitir que no sabe**. ¿Tu prompt lo logra?

In [ ]:
for p in [p for p in preguntas if p["tipo"] == "fuera"]:
    r = rag(p["pregunta"])
    print("¿Admitió no saber?", "✅" if NO_SE.lower()[:20] in r["respuesta"].lower() else "❌", "\n")

## 7. Información contradictoria

Los documentos del curso incluyen a propósito datos desactualizados (avisos, tarifas anteriores, API v1).
¿Tu RAG elige el dato vigente?

In [ ]:
for p in [p for p in preguntas if p["tipo"] == "conflicto"]:
    rag(p["pregunta"], k=4)
    print("   Esperado:", p["respuesta_esperada"], "\n")

## 8. Varios documentos + filtro por metadata

Indexa los tres documentos en el mismo store y usa `filtro` para restringir la búsqueda a una fuente.

In [ ]:
for nombre in ["cafeteria", "streamly", "veterinaria"]:
    if nombre != DOCUMENTO:
        ingestar(nombre, store)
print("Total de chunks en el store:", len(store))

rag("¿Cuál es el horario de atención?")                                   # ¿de qué documento responde?
rag("¿Cuál es el horario de atención?", filtro={"source": "veterinaria"})

## 9. Tu propio ChatGPT 🎉

Chatea con tu RAG. Escribe `salir` para terminar.

In [ ]:
CHAT_INTERACTIVO = False   # cambia a True para chatear

while CHAT_INTERACTIVO:
    pregunta = input("Tú: ").strip()
    if pregunta.lower() in {"salir", "exit", ""}:
        break
    rag(pregunta)

## Preguntas de cierre

1. ¿Qué componente de tu pipeline cambiarías primero si las respuestas fueran malas?
2. ¿Qué pasaría si mañana recibes los documentos de otra empresa? ¿Qué piezas cambian?
3. ¿Por qué es útil mostrar las citas `[id]` en la respuesta?